# 01 Data Exploration & Sanity Verification

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        os.path.join(dirname, filename)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import h5py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt


# Inspect the directory structure inside /kaggle/input
print("Mounted datasets under /kaggle/input:")
for item in os.listdir('/kaggle/input'):
    print(" -", item)

# Search for the directory containing .h5 slice files
target_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'ACDC_training_slices' in dirs:
        target_dir = os.path.join(root, 'ACDC_training_slices')
        break
    elif any(f.endswith('.h5') for f in files):
        target_dir = root
        break

print(f"\nResolved training slices directory: {target_dir}")

In [ ]:
import os
import h5py
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

def find_slices_dir(search_root="/kaggle/input"):
    """Recursively search for the ACDC_training_slices directory."""
    for root, dirs, files in os.walk(search_root):
        if 'ACDC_training_slices' in dirs:
            return os.path.join(root, 'ACDC_training_slices')
    for root, dirs, files in os.walk(search_root):
        if any(f.endswith('.h5') for f in files):
            return root
    raise FileNotFoundError(f"Could not find any directory containing .h5 files under {search_root}")

class ACDC2DDataset(Dataset):
    def __init__(self, target_size=(256, 256)):
        self.slice_dir = find_slices_dir()
        self.target_size = target_size
        self.sample_list = sorted([f for f in os.listdir(self.slice_dir) if f.endswith('.h5')])
        print(f"Successfully loaded {len(self.sample_list)} slices from: {self.slice_dir}")
        
    def __len__(self):
        return len(self.sample_list)
        
    def __getitem__(self, idx):
        file_name = self.sample_list[idx]
        file_path = os.path.join(self.slice_dir, file_name)
        
        with h5py.File(file_path, 'r') as h5f:
            image = h5f['image'][:]
            label = h5f['label'][:]
        
        # 1. Convert to PyTorch tensors and add dummy batch/channel dimensions (1, 1, H, W)
        # Interpolate function requires a 4D tensor.
        img_tensor = torch.tensor(image, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        lbl_tensor = torch.tensor(label, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        
        # 2. Apply Resizing to enforce uniform dimensions
        # Use Bilinear for the MRI Image to keep it smooth
        img_resized = F.interpolate(img_tensor, size=self.target_size, mode='bilinear', align_corners=False)
        
        # Use Nearest Neighbor for the Mask to preserve exact integer classes (0, 1, 2, 3)
        lbl_resized = F.interpolate(lbl_tensor, size=self.target_size, mode='nearest')
        
        # 3. Strip the dummy batch dimension
        # Final image shape: [1, 256, 256] | Final label shape: [256, 256]
        final_image = img_resized.squeeze(0)
        final_label = lbl_resized.squeeze(0).squeeze(0).long()
        
        return final_image, final_label

# Initialize Dataset and DataLoader
train_dataset = ACDC2DDataset(target_size=(256, 256))
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# Verify the first batch
images, labels = next(iter(train_loader))
print(f"Success! Batch images shape: {images.shape}")
print(f"Success! Batch labels shape: {labels.shape}")

# Plot to verify structural integrity is maintained after resize
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("Resized MRI Slice (256x256)")
plt.imshow(images[0].squeeze(), cmap='gray')

plt.subplot(1, 2, 2)
plt.title("Resized Ground Truth Mask (256x256)")
plt.imshow(labels[0], cmap='jet')
plt.show()

In [ ]:
# Inspect the keys and attributes of a single .h5 file
sample_path = os.path.join(train_dataset.slice_dir, train_dataset.sample_list[0])
with h5py.File(sample_path, 'r') as h5f:
    print(f"File: {train_dataset.sample_list[0]}")
    print("Keys inside the HDF5 file:", list(h5f.keys()))
    print("Attributes (metadata):", dict(h5f.attrs))